In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import os
import zipfile

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


In [4]:
zip_path = '/content/drive/MyDrive/preprocessed_photos.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [6]:
train_df = pd.read_csv('/content/train_data.csv')

#  Preserve subfolder in image path
train_df['full_path'] = train_df['preprocessed_path'].apply(
    lambda x: os.path.join(extract_path, x)
)




Sampling the dataset

In [7]:
samples_per_class = 1000

sampled_df = train_df.groupby('label').apply(
    lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42)
).reset_index(drop=True)

print(sampled_df['label'].value_counts())

# Confirm file paths exist
print(f"Found {sampled_df['full_path'].apply(os.path.exists).sum()} / {len(sampled_df)} files")



label
drink      1000
food       1000
inside     1000
menu       1000
outside    1000
Name: count, dtype: int64
Found 5000 / 5000 files


<ipython-input-7-d58eba9e2b14>:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = train_df.groupby('label').apply(


Train/Validation Split

In [8]:
train_data, val_data = train_test_split(
    sampled_df, test_size=0.2, stratify=sampled_df['label'], random_state=42
)



Image Data Generators

In [9]:
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    train_data,
    x_col='full_path',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

val_gen = datagen.flow_from_dataframe(
    val_data,
    x_col='full_path',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)



Found 4000 validated image filenames belonging to 5 classes.
Found 1000 validated image filenames belonging to 5 classes.


Building & Compiling Base VGG16 Model

In [10]:
base_model = VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()



58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,301 (56.64 MB)

 Trainable params: 132,613 (518.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [11]:
sampled_df[['preprocessed_path', 'full_path']].head(10)


,preprocessed_path,full_path
0,preprocessed_photos/GLnI3TTOxDsBpyW3iZD7BA.jpg,/content/dataset/preprocessed_photos/GLnI3TTOx...
1,preprocessed_photos/QgGPIE1JFfeGauBv2oLo4Q.jpg,/content/dataset/preprocessed_photos/QgGPIE1JF...
2,preprocessed_photos/F9SMBGoEqhjrCfQaS4ZA3A.jpg,/content/dataset/preprocessed_photos/F9SMBGoEq...
3,preprocessed_photos/XVJWa1b43o3735iSccCEyQ.jpg,/content/dataset/preprocessed_photos/XVJWa1b43...
4,preprocessed_photos/UB_PfPCbaRruk3-gFZT4Wg.jpg,/content/dataset/preprocessed_photos/UB_PfPCba...
5,preprocessed_photos/7Z2LAmUeA-gBH-RRXQX0QQ.jpg,/content/dataset/preprocessed_photos/7Z2LAmUeA...
6,preprocessed_photos/QVx-ZIEKVm2lU-tASxvtmQ.jpg,/content/dataset/preprocessed_photos/QVx-ZIEKV...
7,preprocessed_photos/mgsvILdf31eN1A-uItV1tg.jpg,/content/dataset/preprocessed_photos/mgsvILdf3...
8,preprocessed_photos/3r_J0_OotZcpgZ1Z-kHJcA.jpg,/content/dataset/preprocessed_photos/3r_J0_Oot...
9,preprocessed_photos/UyfhVTX5l0IsAZOZBxvNIA.jpg,/content/dataset/preprocessed_photos/UyfhVTX5l...


Training Frozen Model

In [12]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)]
)



/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3000s 24s/step - accuracy: 0.2770 - loss: 1.6488 - val_accuracy: 0.6990 - val_loss: 1.1907
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2968s 24s/step - accuracy: 0.5227 - loss: 1.2122 - val_accuracy: 0.7400 - val_loss: 0.9496
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3111s 24s/step - accuracy: 0.6465 - loss: 0.9709 - val_accuracy: 0.7460 - val_loss: 0.8272
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3030s 24s/step - accuracy: 0.6947 - loss: 0.8468 - val_accuracy: 0.7640 - val_loss: 0.7504
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2943s 24s/step - accuracy: 0.7244 - loss: 0.7752 - val_accuracy: 0.7670 - val_loss: 0.7002


Fine-Tuning Last VGG Layers

In [13]:
for layer in base_model.layers[-4:]:  # fine-tuning last 4 layers
    layer.trainable = True

model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)]
)


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3584s 29s/step - accuracy: 0.7723 - loss: 0.6448 - val_accuracy: 0.8090 - val_loss: 0.4874
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3601s 29s/step - accuracy: 0.8248 - loss: 0.4730 - val_accuracy: 0.8400 - val_loss: 0.4064
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3562s 29s/step - accuracy: 0.8513 - loss: 0.3995 - val_accuracy: 0.8480 - val_loss: 0.4027
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3516s 28s/step - accuracy: 0.8750 - loss: 0.3536 - val_accuracy: 0.8620 - val_loss: 0.3551
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3512s 28s/step - accuracy: 0.8896 - loss: 0.3013 - val_accuracy: 0.8770 - val_loss: 0.3160


Saving the final model

In [14]:
model.save('/content/vgg16_finetuned_model.h5')
